# 03 — Coupling as a training loss

Train a tiny linear generator to match the coupling structure of a target
distribution using `CouplingLoss` in `matrix` mode.

In [ ]:
import torch
import matplotlib.pyplot as plt
from couplnorm import coupling_from_samples, CouplingLoss

torch.manual_seed(0)

def coupled_field(B, N, sigma=1.0):
    z = 1.0 + sigma * torch.randn(B, 1)
    return z * torch.randn(B, N)

N = 16

## Build a target and a generator

In [ ]:
target_data = coupled_field(20_000, N, sigma=1.5)
target_C = coupling_from_samples(target_data).item()
print(f'target C = {target_C:.4f}')

loss_fn = CouplingLoss.from_data(target_data, target_type='matrix')
G = torch.nn.Linear(8, N, bias=False)
opt = torch.optim.Adam(G.parameters(), lr=5e-3)

## Train

In [ ]:
losses, C_traj = [], []
for step in range(400):
    z = torch.randn(512, 8)
    L = loss_fn(G(z))
    opt.zero_grad(); L.backward(); opt.step()
    losses.append(L.item())
    if step % 10 == 0:
        with torch.no_grad():
            C_now = coupling_from_samples(G(torch.randn(2000, 8))).item()
        C_traj.append((step, C_now))
steps, Cs = zip(*C_traj)

## Plot training

In [ ]:
from couplnorm.plotting import plot_training
plot_training(losses, steps, Cs, target_C)
plt.show()

## Sanity check

In [ ]:
with torch.no_grad():
    C_final = coupling_from_samples(G(torch.randn(20_000, 8))).item()
print(f'final generator C = {C_final:.4f}')
print(f'target C          = {target_C:.4f}')
print(f'gap               = {abs(C_final - target_C):.4f}')